# سبق 11 - ماڈل کانٹیکسٹ پروٹوکول (MCP)

**ماڈل کانٹیکسٹ پروٹوکول (MCP)** ایک کھلا معیار ہے جو ایجنٹوں کو رن ٹائم پر ٹولز، وسائل، اور ڈیٹا ذرائع کو متحرک طور پر دریافت کرنے اور استعمال کرنے کے قابل بناتا ہے۔ سخت کوڈنگ کے بجائے، MCP ایجنٹوں کو بیرونی سرورز سے جڑنے دیتا ہے جو مانگ پر صلاحیتیں پیش کرتے ہیں۔

اس سبق میں، آپ سیکھیں گے:
- MCP کیا ہے اور ایجنٹ سسٹمز کے لیے یہ کیوں اہم ہے
- MCP کلائنٹ-سرور آرکیٹیکچر کیسے کام کرتا ہے
- MCP انداز کی ٹول دریافت استعمال کرنے والے ایجنٹ کیسے بنائیں


## ترتیب

**ضروریات:**
- Microsoft Foundry پروجیکٹ جس میں ایک تعینات ماڈل ہو
- توثیق کے لیے `az login` چلائیں


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ماڈل کانٹیکسٹ پروٹوکول (MCP) کیا ہے؟

MCP ایک معیاری طریقہ کار متعین کرتا ہے جس کے ذریعے AI ایجنٹس خارجی آلات اور ڈیٹا ذرائع کو دریافت اور ان کے ساتھ تعامل کرتے ہیں:

- **MCP سرور**: آلات، وسائل، اور پرامپٹس کو ایک معیاری پروٹوکول کے ذریعے مہیا کرتا ہے
- **MCP کلائنٹ**: ایجنٹ رن ٹائم جو سرورز سے جڑتا ہے اور دستیاب صلاحیتوں کو دریافت کرتا ہے
- **متحرک دریافت**: ایجنٹس کو ہارڈ کوڈ شدہ آلات کی ضرورت نہیں — وہ رن ٹائم پر دستیاب آلات کو دریافت کرتے ہیں

یہ قابل توسیع ایجنٹ سسٹمز بنانے کے لئے طاقتور ہے جہاں نئی صلاحیتیں ایجنٹ کوڈ کو تبدیل کیے بغیر شامل کی جا سکتی ہیں۔


## MCP کیسے کام کرتا ہے

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. ایجنٹ (MCP کلائنٹ) ایک MCP سرور سے جڑتا ہے
2. سرور دستیاب ٹولز اور ان کے اسکیموں کی فہرست کے ساتھ جواب دیتا ہے
3. ایجنٹ پھر اپنے استدلال کے دوران کسی بھی دریافت شدہ ٹول کو کال کر سکتا ہے
4. نتائج اسی پروٹوکول کے ذریعے واپس آتے ہیں


## ایم سی پی ٹول کی دریافت کی نقلی مشق  

چونکہ ایک حقیقی ایم سی پی سرور کے لئے ایک چل رہا سرور عمل درکار ہوتا ہے، ہم یہاں `@tool` فنکشنز استعمال کریں گے جو کہ ایک ایم سی پی سے جڑے ہوئے رہائشی سروس کی تعریف کی نقل کرتے ہیں۔  

پیداوار میں، یہ ٹولز مقامی سطح پر متعین ہونے کے بجائے، متحرک طور پر ایم سی پی سرور سے دریافت کیے جائیں گے۔  


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-طرز کے اوزار کے ساتھ ایک ایجنٹ کی تعمیر


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## پروڈکشن میں MCP

پروڈکشن ماحول میں، MCP طاقتور پیٹرنز کو فعال کرتا ہے:

- **متحرک ٹول دریافت**: ایجنٹس MCP سرورز سے جڑتے ہیں اور رن ٹائم پر ٹولز دریافت کرتے ہیں
- **آزاد ساخت**: ٹول فراہم کرنے والے ایجنٹ سے آزادانہ طور پر اپ ڈیٹ کر سکتے ہیں
- **تنظیم کے پار اشتراک**: ٹیمیں MCP سرورز کے ذریعے صلاحیتیں ظاہر کر سکتی ہیں جنہیں کوئی بھی ایجنٹ استعمال کر سکتا ہے
- **مائیکروسافٹ ایجنٹ فریم ورک کی حمایت**: MAF میں `mcp` انٹیگریشن کے ذریعے بلٹ ان MCP کلائنٹ سپورٹ موجود ہے

MAF کے ساتھ اصلی MCP سرور استعمال کرنے کے لیے، آپ `hosted_mcp_tool()` یا MCP کلائنٹ انٹیگریشن کے ذریعے جڑیں گے۔

**مزید جانیں:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## خلاصہ

اس سبق میں، آپ نے سیکھا:
- **MCP** ایجنٹس اور ٹول فراہم کنندگان کے درمیان متحرک ٹول دریافت کے لئے ایک اوپن اسٹینڈرڈ ہے
- **کلائنٹ-سرور فن تعمیر** ایجنٹس کو رن ٹائم پر صلاحیتیں دریافت کرنے دیتا ہے
- MCP ایسی **قابل توسیع، مربوط ایجنٹ سسٹمز** کو ممکن بناتا ہے جہاں ٹولز کو بغیر کوڈ میں تبدیلی کے شامل کیا جا سکتا ہے
- مائیکروسافٹ ایجنٹ فریم ورک پیداوار کے استعمال کے لئے **بلٹ ان MCP سپورٹ** فراہم کرتا ہے


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
